In [37]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import json

df = pd.read_csv('../data/ufc_clean.csv')
print(f'Rijen: {len(df)}')
print(f'Kolommen: {len(df.columns)}')
df.head()

Rijen: 8520
Kolommen: 99


,RedFighter,BlueFighter,Winner,WeightClass,TitleBout,Method,RedWins,RedLosses,RedWinStreak,RedLoseStreak,...,DifAvgClinchLand,DifAvgGroundLand,DifRecentWins,DifRecentAvgSigStrLand,DifRecentAvgSigStrPct,DifRecentAvgTDLand,DifRecentAvgTDPct,DifRecentAvgSubAtt,DifRecentAvgCtrlSec,DifRecentAvgKD
0,Jack Della Maddalena,Carlos Prates,Blue,Welterweight,False,KO/TKO,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Beneil Dariush,Quillan Salkilld,Blue,Lightweight,False,KO/TKO,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Tim Elliott,Steve Erceg,Blue,Flyweight,False,Decision - Unanimous,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Marwan Rahiki,Ollie Schmid,Red,Featherweight,False,KO/TKO,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Shamil Gaziev,Brando Pericic,Blue,Heavyweight,False,KO/TKO,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
df['target'] = (df['Winner'] == 'Red').astype(int)

exclude = ['RedFighter', 'BlueFighter', 'Winner', 'WeightClass', 'Method', 'target']
feature_cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['float64', 'int64']]

print(f'Aantal features: {len(feature_cols)}')
print(feature_cols)

Aantal features: 93
['RedWins', 'RedLosses', 'RedWinStreak', 'RedLoseStreak', 'RedLongestWinStreak', 'RedKOWins', 'RedSubWins', 'RedDecWins', 'RedTitleBouts', 'RedAvgSigStrLand', 'RedAvgSigStrAtt', 'RedAvgSigStrPct', 'RedAvgTDLand', 'RedAvgTDPct', 'RedAvgSubAtt', 'RedAvgCtrlSec', 'RedAvgKD', 'RedAvgHeadLand', 'RedAvgBodyLand', 'RedAvgLegLand', 'RedAvgDistanceLand', 'RedAvgClinchLand', 'RedAvgGroundLand', 'RedRecentWins', 'RedRecentAvgSigStrLand', 'RedRecentAvgSigStrPct', 'RedRecentAvgTDLand', 'RedRecentAvgTDPct', 'RedRecentAvgSubAtt', 'RedRecentAvgCtrlSec', 'RedRecentAvgKD', 'BlueWins', 'BlueLosses', 'BlueWinStreak', 'BlueLoseStreak', 'BlueLongestWinStreak', 'BlueKOWins', 'BlueSubWins', 'BlueDecWins', 'BlueTitleBouts', 'BlueAvgSigStrLand', 'BlueAvgSigStrAtt', 'BlueAvgSigStrPct', 'BlueAvgTDLand', 'BlueAvgTDPct', 'BlueAvgSubAtt', 'BlueAvgCtrlSec', 'BlueAvgKD', 'BlueAvgHeadLand', 'BlueAvgBodyLand', 'BlueAvgLegLand', 'BlueAvgDistanceLand', 'BlueAvgClinchLand', 'BlueAvgGroundLand', 'BlueRec

In [39]:
X = df[feature_cols].fillna(0)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {len(X_train)} rijen')
print(f'Test set:     {len(X_test)} rijen')

Training set: 6816 rijen
Test set:     1704 rijen


In [40]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f} ({acc*100:.2f}%)')

Accuracy: 0.7788 (77.88%)


In [41]:
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False).head(20)

DifLosses                  0.043869
DifLongestWinStreak        0.032343
DifTitleBouts              0.029082
DifWins                    0.024375
DifAvgSigStrPct            0.022741
DifRecentAvgSigStrPct      0.021358
BlueAvgGroundLand          0.021343
RedAvgSigStrPct            0.018405
DifRecentWins              0.018236
BlueAvgSigStrAtt           0.016846
DifAvgDistanceLand         0.016696
BlueAvgSigStrPct           0.016537
RedWins                    0.016270
BlueRecentAvgSigStrLand    0.016006
RedAvgSigStrAtt            0.015952
RedLosses                  0.015932
DifAvgGroundLand           0.015713
RedAvgCtrlSec              0.015409
BlueRecentAvgSigStrPct     0.015373
RedAvgSigStrLand           0.014158
dtype: float64

In [42]:
import joblib
import json
import os

# Sla op in de root models/ map, niet notebooks/models/
root = os.path.dirname(os.getcwd())
os.makedirs(f"{root}/models", exist_ok=True)

joblib.dump(model, f"{root}/models/model_v4.pkl")

with open(f"{root}/models/feature_names_v4.json", "w") as f:
    json.dump(list(feature_cols), f)

print(f"✅ Model opgeslagen → {root}/models/model_v4.pkl")

✅ Model opgeslagen → /Users/ahmd/Documents/Development/Personal/MMA Fight Predictor/mma-fight-predictor/models/model_v4.pkl
